# Cross-Source Sim-to-Real Sigmoid Validation (H1.1b)

This notebook trains the three best 64x64 ResNet18-pretrained simulator configurations from `strategy_64x64_resnet18` and a real-fixations baseline, then evaluates all models on sigmoid/no-class real sources plus simulated holdout data. All training and validation inputs use the Week-23 fixed-trial inverse-sort/polarity augmentation contract: all sigmoid chunks are kept, one no-class chunk per origin is kept, and every kept chunk is rendered as four variants. The central H1.1b output is `h1_1b_gap_summary.csv`, which reports the gap between simulated-side and real-source balanced accuracy/F1.

In [1]:
# ============================================================================
# Imports, constants, and run configuration
# ============================================================================

import Pkg

ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"
ENV["JULIA_NUM_PRECOMPILE_TASKS"] = "1"

function find_repo_root(start_dir::AbstractString = @__DIR__)
    candidates = unique(normpath.([
        start_dir,
        pwd(),
        joinpath(start_dir, ".."),
        joinpath(start_dir, "..", ".."),
        joinpath(pwd(), ".."),
        joinpath(pwd(), "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "datasets"))
            return candidate
        end
    end
    error("Could not locate repository root from start_dir=$(start_dir), pwd=$(pwd()).")
end

const REPO_ROOT = find_repo_root()
const NOTEBOOK_DIR = joinpath(REPO_ROOT, "notebooks", "data_generation")
const DATA_GENERATION_ENV = NOTEBOOK_DIR
const MODEL_TEST_ENV = joinpath(REPO_ROOT, "notebooks", "model_test")
for env_path in (MODEL_TEST_ENV, DATA_GENERATION_ENV)
    env_path in LOAD_PATH || push!(LOAD_PATH, env_path)
end
Pkg.activate(MODEL_TEST_ENV; io = devnull)

using CairoMakie
using CSV
using DataFrames
using Dates
using Distributions
using Flux
using Flux: onecold, onehotbatch
using JLD2
using JSON3
using MLUtils: DataLoader
using Printf: @sprintf
using Random
using Statistics

try
    using CUDA
    using cuDNN
catch err
    @warn "CUDA/cuDNN could not be loaded; CPU execution will be used." exception = (err, catch_backtrace())
end

include(joinpath(REPO_ROOT, "notebooks", "week_15", "simulation_small_scale_helpers.jl"))
include(joinpath(REPO_ROOT, "notebooks", "utils", "erp_cnn_experiment_utils.jl"))

const SS = SmallScaleERPClassification
const ERPGenS = SmallScaleERPClassification.ERPGen
const CNNUtils = ERPCNNExperimentUtils

const DATASETS_ROOT = joinpath(REPO_ROOT, "datasets")
const WEEK21_SUMMARY_JSON = joinpath(REPO_ROOT, "notebooks", "week_21", "outputs", "week21_labeling_summary", "summary.json")
const TOP_PER_STRATEGY_CSV = joinpath(NOTEBOOK_DIR, "outputs", "strategy_64x64_resnet18", "posthoc_exports", "top_per_strategy.csv")
const OUTPUT_DIR = joinpath(NOTEBOOK_DIR, "outputs", "cross_source_sim_to_real_sigmoid")
const PREVIEW_PATH = joinpath(OUTPUT_DIR, "preview_sim_vs_real_per_source.png")

const TARGET_SIZE = (64, 64)
const PIPELINE_NAME = :gaussian_reference
const LOW_PASS_FACTOR = 75.0f0
const LOWPASS_KERNEL_SIZE = (21, 21)
const FILTER_BORDER = "reflect"

const SIGMOID_CLASS = "sigmoid"
const NO_CLASS = "no_class"
const REAL_BASELINE_SOURCE = "fixations_dataset"
const REAL_BASELINE_SIGMOID_SORT_VARIABLE = "duration"
const TARGET_TRIALS = 200
const NO_CLASS_CHUNKS_PER_ORIGIN = 1

const N_PER_PATTERN = 1000
const SIM_HOLDOUT_PAIRS = 200
const N_REPEATS = 3
const TRAIN_EPOCHS = 8
const TRAIN_BATCHSIZE = 64
const TRAIN_LR = 3f-4
const LABEL_SMOOTHING = 0.02f0
const CLASS_WEIGHTS = Float32[1.0, 1.0]
const PREDICT_BATCHSIZE = 64

const SANITY_BACC_MIN = 0.55
const SANITY_CLASS_BALANCE_FRAC = 0.05
const MINI_TEST_PAIRS = 50
const MIN_GENERATION_THREADS = 16
const BASE_SEED = parse(Int, get(ENV, "CROSS_SOURCE_BASE_SEED", "20260523"))

const REQUESTED_REAL_SOURCE_KEYS = [
    "fixations_dataset",
    "eye_eeg_freeviewing_fixations",
    "erp_core_n170_clean",
    "erp_core_n2pc_clean",
    "02_new_roamm_reading",
]

const SIM_MODEL_SPECS = [
    (model_kind = "sim_broad_random", search_method = "broad_random", candidate_index = 10),
    (model_kind = "sim_latin_hypercube", search_method = "latin_hypercube", candidate_index = 21),
    (model_kind = "sim_monte_carlo", search_method = "monte_carlo", candidate_index = 38),
]

const TRAINING_PROFILE = (
    name = "pretrained_default",
    model_init = :pretrained,
    nepochs = TRAIN_EPOCHS,
    lr = TRAIN_LR,
    batchsize = TRAIN_BATCHSIZE,
    class_weights = CLASS_WEIGHTS,
    label_smoothing = LABEL_SMOOTHING,
)

const REQUIRE_GPU = parse(Bool, get(ENV, "CROSS_SOURCE_REQUIRE_GPU", "true"))
const USE_GPU = (@isdefined CUDA) && CUDA.functional()
REQUIRE_GPU && !USE_GPU && error("CUDA GPU training is required for this experiment. Set CROSS_SOURCE_REQUIRE_GPU=false only for a CPU debug run.")
if USE_GPU
    CUDA.allowscalar(false)
    CUDA.device!(0)
end
const GPU_NAME = USE_GPU ? CUDA.name(CUDA.device()) : "CPU"

device(x) = USE_GPU ? gpu(x) : x

println("REPO_ROOT = ", REPO_ROOT)
println("Output directory = ", OUTPUT_DIR)
println("Threads.nthreads() = ", Threads.nthreads())
println("Device = ", USE_GPU ? "CUDA GPU ($(GPU_NAME))" : "CPU")
println("BASE_SEED = ", BASE_SEED)

REPO_ROOT = /home/benjamin/Dokumente/BA2/
Output directory = /home/benjamin/Dokumente/BA2/notebooks/data_generation/outputs/cross_source_sim_to_real_sigmoid
Threads.nthreads() = 16
Device = CUDA GPU (NVIDIA GeForce RTX 4070)
BASE_SEED = 20260523


In [2]:
# ============================================================================
# Shared utility functions
# ============================================================================

function cellstr(x)
    return ismissing(x) ? "" : String(x)
end

function derived_seed(parts...)
    h = UInt64(BASE_SEED)
    prime = UInt64(0x100000001b3)
    for part in parts
        for b in codeunits(string(part))
            h = xor(h, UInt64(b)) * prime
        end
        h = xor(h, UInt64(0xff)) * prime
    end
    return Int(mod(h, UInt64(typemax(Int) - 1))) + 1
end

function set_all_seeds!(seed::Integer)
    Random.seed!(seed)
    if USE_GPU && isdefined(CUDA, :seed!)
        try
            CUDA.seed!(seed)
        catch err
            @warn "CUDA.seed! failed; continuing after Random.seed!." exception = (err, catch_backtrace())
        end
    end
    return Random.Xoshiro(seed)
end

function cleanup_device!()
    GC.gc()
    USE_GPU && CUDA.reclaim()
    return nothing
end

function ensure_output_dir!()
    mkpath(OUTPUT_DIR)
    return OUTPUT_DIR
end

function mean_or_missing(xs)
    vals = Float64[]
    for x in xs
        if !ismissing(x)
            xf = Float64(x)
            isfinite(xf) && push!(vals, xf)
        end
    end
    return isempty(vals) ? missing : mean(vals)
end

function std_or_zero(xs)
    vals = Float64[]
    for x in xs
        if !ismissing(x)
            xf = Float64(x)
            isfinite(xf) && push!(vals, xf)
        end
    end
    isempty(vals) && return missing
    length(vals) == 1 && return 0.0
    return std(vals)
end

function binary_metrics(y_pred::Vector{Int}, y_true::Vector{Int})
    classes = (0, 1)
    recalls = Float64[]
    precisions = Float64[]
    f1s = Float64[]
    for cls in classes
        tp = count(i -> y_pred[i] == cls && y_true[i] == cls, eachindex(y_true))
        fp = count(i -> y_pred[i] == cls && y_true[i] != cls, eachindex(y_true))
        fn = count(i -> y_pred[i] != cls && y_true[i] == cls, eachindex(y_true))
        precision = (tp + fp) == 0 ? 0.0 : tp / (tp + fp)
        recall = (tp + fn) == 0 ? 0.0 : tp / (tp + fn)
        f1 = (precision + recall) == 0 ? 0.0 : 2 * precision * recall / (precision + recall)
        push!(precisions, precision)
        push!(recalls, recall)
        push!(f1s, f1)
    end
    return (
        balanced_accuracy = mean(recalls),
        macro_f1 = mean(f1s),
        precision = mean(precisions),
        recall = mean(recalls),
    )
end

function required_predicted_class_count(n_validation::Integer)
    return ceil(Int, SANITY_CLASS_BALANCE_FRAC * Int(n_validation))
end

function prediction_collapsed(y_pred::Vector{Int})
    n_validation = length(y_pred)
    min(count(==(0), y_pred), count(==(1), y_pred)) < required_predicted_class_count(n_validation)
end

function smooth_onehot(y_oh::AbstractMatrix{<:Real}, epsilon::Real)
    epsilon <= 0 && return Array{Float32}(y_oh)
    n_classes = size(y_oh, 1)
    return Float32.((1 - epsilon) .* y_oh .+ epsilon / n_classes)
end

function weighted_logitcrossentropy(logits, y_oh, class_weights)
    logp = Flux.logsoftmax(logits; dims = 1)
    return mean(-sum(class_weights .* y_oh .* logp; dims = 1))
end

weighted_logitcrossentropy (generic function with 1 method)

In [3]:
# ============================================================================
# Real-source discovery and shared preprocessing
# ============================================================================

function dataset_paths(dataset_key::AbstractString)
    dir = joinpath(DATASETS_ROOT, dataset_key)
    return (
        dir = dir,
        events_path = joinpath(dir, "events.jld2"),
        labels_path = joinpath(dir, "labels.jld2"),
        signals_dir = joinpath(dir, "signals"),
    )
end

function signal_path(dataset_key::AbstractString, channel_name::AbstractString)
    return joinpath(dataset_paths(dataset_key).signals_dir, string(channel_name, ".jld2"))
end

function load_dataset_tables(dataset_key::AbstractString)
    paths = dataset_paths(dataset_key)
    isfile(paths.events_path) || error("Missing events file: $(paths.events_path)")
    isfile(paths.labels_path) || error("Missing labels file: $(paths.labels_path)")
    events = JLD2.load(paths.events_path, "events")
    labels = JLD2.load(paths.labels_path, "labels")
    metadata = JLD2.load(paths.events_path, "metadata")
    labels.dataset_key = fill(String(dataset_key), nrow(labels))
    labels.erp_class = cellstr.(labels.erp_class)
    labels.sort_variable = cellstr.(labels.sort_variable)
    labels.channel_name = cellstr.(labels.channel_name)
    return events, labels, metadata
end

function load_signal_for_label(dataset_key::AbstractString, channel_name::AbstractString, cache::Dict{Tuple{String, String}, Matrix{Float32}})
    key = (String(dataset_key), String(channel_name))
    return get!(cache, key) do
        path = signal_path(dataset_key, channel_name)
        isfile(path) || error("Missing signal file for $(dataset_key)/$(channel_name): $(path)")
        Matrix{Float32}(JLD2.load(path, "data_time_trials"))
    end
end

const AUGMENTATION_VARIANTS = (
    (name = :reference, label = "normal sort, normal polarity", inverse_sort = false, inverse_polarity = false),
    (name = :inverse_sort, label = "inverse sort, normal polarity", inverse_sort = true, inverse_polarity = false),
    (name = :inverse_polarity, label = "normal sort, inverse polarity", inverse_sort = false, inverse_polarity = true),
    (name = :inverse_sort_inverse_polarity, label = "inverse sort, inverse polarity", inverse_sort = true, inverse_polarity = true),
)
const AUGMENTATION_VARIANT_COUNT = length(AUGMENTATION_VARIANTS)

function is_valid_sort_value(v)
    (ismissing(v) || v === nothing) && return false
    if v isa Real
        return isfinite(Float64(v))
    end
    return !isempty(strip(string(v)))
end

function valid_sort_mask(events::DataFrame, sort_col::Symbol)
    sort_col in propertynames(events) || error("Sort column $(sort_col) not found in events table.")
    return [is_valid_sort_value(v) for v in events[!, sort_col]]
end

function filtered_events_and_data_for_sort(data_time_trials::AbstractMatrix, events::DataFrame, sort_col::Symbol)
    size(data_time_trials, 2) == nrow(events) || error("Trial count mismatch between signal matrix and events table.")
    keep = valid_sort_mask(events, sort_col)
    n_kept = count(identity, keep)
    n_kept > 0 || error("No valid sort values for $(sort_col).")
    return (
        events = events[keep, :],
        data_time_trials = Float32.(data_time_trials[:, keep]),
        n_filtered_out = length(keep) - n_kept,
    )
end

function sorted_order_for_variant(events_trials::DataFrame, sort_col::Symbol; inverse_sort::Bool = false)
    sort_col in propertynames(events_trials) || error("Sort column $(sort_col) missing.")
    sortvals = CNNUtils.sortvalues_from(events_trials, sort_col)
    order = sortperm(sortvals)
    inverse_sort && reverse!(order)
    return order
end

function fill_remainder_indices(order::Vector{Int}, remainder_idxs::Vector{Int}, target_trials::Int,
        source_row_id::Int, sort_variable::AbstractString)
    needed = target_trials - length(remainder_idxs)
    needed <= 0 && return Int[]
    used = Set(remainder_idxs)
    candidates = [idx for idx in order if !(idx in used)]
    if length(candidates) < needed
        candidates = copy(order)
    end
    seed = derived_seed(:target_trial_remainder, source_row_id, sort_variable, target_trials, length(order))
    rng = MersenneTwister(seed)
    return shuffle(rng, candidates)[1:needed]
end

function target_trial_mod_chunks(order::Vector{Int}, target_trials::Int;
        source_row_id::Int,
        sort_variable::AbstractString)
    n = length(order)
    target_trials <= 0 && error("target_trials must be positive")
    n < target_trials && error("Cannot make fixed-size chunks of $(target_trials) from only $(n) trials")
    full_chunk_count = div(n, target_trials)
    remainder_count = rem(n, target_trials)
    full_chunk_count >= 1 || error("Cannot build mod chunks for n=$(n), target_trials=$(target_trials).")

    full_bins = [Int[] for _ in 1:full_chunk_count]
    remainder = Int[]
    rank = 1
    while rank <= n
        progressed = false
        for bin in full_bins
            if length(bin) < target_trials && rank <= n
                push!(bin, order[rank])
                rank += 1
                progressed = true
            end
        end
        if remainder_count > 0 && length(remainder) < remainder_count && rank <= n
            push!(remainder, order[rank])
            rank += 1
            progressed = true
        end
        progressed || break
    end

    chunks = NamedTuple[]
    for (chunk_index, idxs) in enumerate(full_bins)
        push!(chunks, (
            chunk_index = Int(chunk_index),
            chunk_count = 0,
            chunk_role = "full_mod_split",
            full_mod_split_k = Int(full_chunk_count),
            remainder_trials = Int(remainder_count),
            unique_trial_count_before_fill = Int(length(unique(idxs))),
            reused_fill_count = 0,
            filler_indices = Int[],
            trial_indices = copy(idxs),
        ))
    end

    if remainder_count > 0
        filler = fill_remainder_indices(order, remainder, target_trials, source_row_id, sort_variable)
        idxs = vcat(remainder, filler)
        @assert length(idxs) == target_trials "Filled remainder chunk did not reach target_trials."
        push!(chunks, (
            chunk_index = Int(length(chunks) + 1),
            chunk_count = 0,
            chunk_role = "distributed_remainder_filled",
            full_mod_split_k = Int(full_chunk_count),
            remainder_trials = Int(remainder_count),
            unique_trial_count_before_fill = Int(length(unique(remainder))),
            reused_fill_count = Int(length(filler)),
            filler_indices = filler,
            trial_indices = idxs,
        ))
    end

    chunk_count = length(chunks)
    return [merge(c, (chunk_count = chunk_count,)) for c in chunks]
end

function no_class_chunk_indices(chunks, source_row_id::Int)
    isempty(chunks) && return Int[]
    keep_n = min(NO_CLASS_CHUNKS_PER_ORIGIN, length(chunks))
    start = mod(source_row_id - 1, length(chunks)) + 1
    return [mod(start + j - 2, length(chunks)) + 1 for j in 1:keep_n]
end

function chunk_indices_for_label(binary_label::Integer, chunks, source_row_id::Int)
    return Int(binary_label) == 1 ? collect(eachindex(chunks)) : no_class_chunk_indices(chunks, source_row_id)
end

function zscore_timepoints_local(data_time_trials::AbstractMatrix)
    x = Float32.(data_time_trials)
    mu = mean(x; dims = 2)
    sigma = std(x; dims = 2, corrected = true)
    sigma_safe = ifelse.(Float32.(sigma) .== 0f0, 1f0, Float32.(sigma))
    return Float32.((x .- Float32.(mu)) ./ sigma_safe)
end

function preprocess_ordered_variant(data_time_trials_ordered::AbstractMatrix)
    data_z = zscore_timepoints_local(data_time_trials_ordered)
    img_trials_time = Float32.(permutedims(data_z, (2, 1)))
    return CNNUtils.apply_pipeline_to_image(
        img_trials_time;
        pipeline_name = PIPELINE_NAME,
        target_size = TARGET_SIZE,
        low_pass_sigma = LOW_PASS_FACTOR,
        lowpass_kernel_size = LOWPASS_KERNEL_SIZE,
        filter_border = FILTER_BORDER,
    )
end

function preprocess_ordered_chunk_variant(data_time_trials_ordered::AbstractMatrix, augmentation)
    data_variant = augmentation.inverse_sort ? reverse(Float32.(data_time_trials_ordered), dims = 2) : copy(Float32.(data_time_trials_ordered))
    augmentation.inverse_polarity && (data_variant .*= -1f0)
    return preprocess_ordered_variant(data_variant)
end

function preprocess_fixed_trial_image(data_time_trials::AbstractMatrix, events_trials::DataFrame, sort_col::Symbol, augmentation)
    size(data_time_trials, 2) == nrow(events_trials) || error("Trial count mismatch between matrix and events table.")
    order = sorted_order_for_variant(events_trials, sort_col; inverse_sort = Bool(augmentation.inverse_sort))
    data_ordered = Float32.(data_time_trials[:, order])
    augmentation.inverse_polarity && (data_ordered .*= -1f0)
    return preprocess_ordered_variant(data_ordered)
end

function eligible_sigmoid_sources_from_summary()
    isfile(WEEK21_SUMMARY_JSON) || error("Missing Week-21 summary: $(WEEK21_SUMMARY_JSON)")
    summary = JSON3.read(read(WEEK21_SUMMARY_JSON, String))
    rows = NamedTuple[]
    for row in summary.dataset_summary
        classes = split(String(row.pattern_classes), ';')
        has_sigmoid_class = SIGMOID_CLASS in classes
        has_sigmoid_class || continue
        push!(rows, (
            dataset_key = String(row.dataset_key),
            dataset_label = String(row.dataset_label),
            total_labeled = Int(row.total_labeled),
            pattern_labeled = Int(row.pattern_labeled),
            no_class_labeled = Int(row.no_class_labeled),
            pattern_classes = String(row.pattern_classes),
        ))
    end
    df = DataFrame(rows)
    isempty(df) || sort!(df, :dataset_key)
    return df
end

function source_validation_sort_variables(labels::DataFrame, dataset_key::AbstractString)
    if dataset_key == REAL_BASELINE_SOURCE
        return [REAL_BASELINE_SIGMOID_SORT_VARIABLE]
    end
    vars = sort(unique(String.(labels.sort_variable[labels.erp_class .== SIGMOID_CLASS])))
    isempty(vars) && error("No sigmoid sort variables in $(dataset_key).")
    return vars
end

function select_validation_labels(labels::DataFrame, dataset_key::AbstractString)
    sort_vars = source_validation_sort_variables(labels, dataset_key)
    sort_var_set = Set(sort_vars)
    keep = [
        String(row.erp_class) in (SIGMOID_CLASS, NO_CLASS) && String(row.sort_variable) in sort_var_set
        for row in eachrow(labels)
    ]
    selected = copy(labels[keep, :])
    selected.binary_label = Int.(selected.erp_class .== SIGMOID_CLASS)
    sort!(selected, [:erp_class, :sort_variable, :channel_name])
    selected.source_row_id = collect(1:nrow(selected))
    count(==(1), selected.binary_label) > 0 || error("No sigmoid labels kept for $(dataset_key).")
    count(==(0), selected.binary_label) > 0 || error("No no_class labels kept for $(dataset_key).")
    return selected, sort_vars
end

function materialize_real_validation_source(dataset_key::AbstractString; target_trials::Int = TARGET_TRIALS)
    events, labels, metadata = load_dataset_tables(dataset_key)
    selected, sort_vars = select_validation_labels(labels, dataset_key)
    rows = NamedTuple[]
    images = Matrix{Float32}[]
    signal_cache = Dict{Tuple{String, String}, Matrix{Float32}}()
    base_sample_id = 0

    for row in eachrow(selected)
        data_time_trials = load_signal_for_label(dataset_key, row.channel_name, signal_cache)
        sort_col = Symbol(String(row.sort_variable))
        filtered = filtered_events_and_data_for_sort(data_time_trials, events, sort_col)
        nrow(filtered.events) >= target_trials || error("$(dataset_key)/$(row.channel_name)/$(row.sort_variable) has only $(nrow(filtered.events)) valid trials; target_trials=$(target_trials).")
        order = sorted_order_for_variant(filtered.events, sort_col)
        chunks = target_trial_mod_chunks(
            order,
            target_trials;
            source_row_id = Int(row.source_row_id),
            sort_variable = String(row.sort_variable),
        )
        chunk_idxs = chunk_indices_for_label(Int(row.binary_label), chunks, Int(row.source_row_id))

        for chunk_idx in chunk_idxs
            chunk = chunks[chunk_idx]
            idxs = chunk.trial_indices
            events_part = filtered.events[idxs, :]
            data_part = filtered.data_time_trials[:, idxs]
            base_sample_id += 1
            mod_variant = chunk.reused_fill_count == 0 ?
                @sprintf("modtarget_%04d_part%03d", target_trials, chunk.chunk_index) :
                @sprintf("modtarget_%04d_remainder%03d_fill%03d", target_trials, chunk.chunk_index, chunk.reused_fill_count)

            for (augmentation_variant_index, augmentation) in enumerate(AUGMENTATION_VARIANTS)
                img = preprocess_fixed_trial_image(data_part, events_part, sort_col, augmentation)
                push!(rows, (
                    sample_id = length(rows) + 1,
                    base_sample_id = Int(base_sample_id),
                    label_index = Int(row.source_row_id),
                    dataset_key = String(dataset_key),
                    channel_name = String(row.channel_name),
                    sort_variable = String(row.sort_variable),
                    erp_class = String(row.erp_class),
                    binary_label = Int(row.binary_label),
                    origin_n_trials = nrow(events),
                    n_trials = nrow(filtered.events),
                    n_filtered_out = Int(filtered.n_filtered_out),
                    n_timepoints = size(filtered.data_time_trials, 1),
                    sampling_rate_hz = Float64(metadata["sampling_rate_hz"]),
                    target_trials = Int(target_trials),
                    mod_variant = mod_variant,
                    chunk_index = Int(chunk.chunk_index),
                    chunk_count = Int(chunk.chunk_count),
                    chunk_role = String(chunk.chunk_role),
                    reused_fill_count = Int(chunk.reused_fill_count),
                    augmentation_variant_index = Int(augmentation_variant_index),
                    augmentation_name = String(augmentation.name),
                    augmentation_label = String(augmentation.label),
                    inverse_sort = Bool(augmentation.inverse_sort),
                    inverse_polarity = Bool(augmentation.inverse_polarity),
                    variant = String(augmentation.name),
                ))
                push!(images, img)
            end
        end
    end

    out = DataFrame(rows)
    out.processed_img = images
    @assert all(size.(out.processed_img) .== Ref(TARGET_SIZE))
    return out, sort_vars
end

eligible_sources_df = eligible_sigmoid_sources_from_summary()
missing_requested = setdiff(REQUESTED_REAL_SOURCE_KEYS, String.(eligible_sources_df.dataset_key))
isempty(missing_requested) || error("Requested H1.1b validation source(s) are not sigmoid-positive in summary.json: $(missing_requested)")

real_validation_sets = Dict{String, NamedTuple}()
source_rows = NamedTuple[]
for dataset_key in REQUESTED_REAL_SOURCE_KEYS
    df, sort_vars = materialize_real_validation_source(dataset_key)
    X = CNNUtils.images_to_tensor(df.processed_img)
    y = Int.(df.binary_label)
    real_validation_sets[dataset_key] = (df = df, X = X, y = y, sort_variables = sort_vars)
    push!(source_rows, (
        validation_source = dataset_key,
        sort_variables = join(sort_vars, ";"),
        n_validation = length(y),
        n_sigmoid = count(==(1), y),
        n_no_class = count(==(0), y),
    ))
end
real_source_summary_df = DataFrame(source_rows)
show(real_source_summary_df, allrows = true, allcols = true)
println()

5×5 DataFrame
 Row │ validation_source              sort_variables                     n_validation  n_sigmoid  n_no_class 
     │ String                         String                             Int64         Int64      Int64      
─────┼───────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ fixations_dataset              duration                                   3484       3224         260
   2 │ eye_eeg_freeviewing_fixations  fixation_duration_ms                        196        144          52
   3 │ erp_core_n170_clean            reaction_time_ms                            284        192          92
   4 │ erp_core_n2pc_clean            reaction_time_ms                            300        192         108
   5 │ 02_new_roamm_reading           fixation_duration;fixation_durat…           832        400         432


In [4]:
# ============================================================================
# Fixation-matched simulator base config and top candidate configs
# ============================================================================

fix_events, fix_labels_for_dims, fix_metadata = load_dataset_tables(REAL_BASELINE_SOURCE)
const REAL_N_TRIALS = nrow(fix_events)
const REAL_N_TIMEPOINTS = Int(fix_metadata["n_timepoints_post"])
const REAL_SAMPLING_RATE = Float64(fix_metadata["sampling_rate_hz"])
const REAL_EPOCH_DURATION_S = Float64(REAL_N_TIMEPOINTS - 1) / REAL_SAMPLING_RATE

println("Reference dimensions before downscale: trials=", REAL_N_TRIALS,
    ", timepoints=", REAL_N_TIMEPOINTS,
    ", sampling_rate=", REAL_SAMPLING_RATE,
    ", epoch_duration_s=", REAL_EPOCH_DURATION_S)

function install_deterministic_erpgen_rng!()
    @eval SmallScaleERPClassification.ERPGen begin
        @inline fresh_seed(rng::Random.AbstractRNG = Random.default_rng()) = rand(rng, UInt64)
        @inline fresh_rng(rng::Random.AbstractRNG = Random.default_rng()) = Random.Xoshiro(fresh_seed(rng))
        @inline time_seeded_rand() = rand(Random.default_rng())
        @inline time_seeded_rand(sampler) = rand(Random.default_rng(), sampler)
        function time_seeded_rand(sampler, dims::Integer...)
            return rand(Random.default_rng(), sampler, dims...)
        end
        @inline time_seeded_randperm(n::Integer) = randperm(Random.default_rng(), n)
        @inline time_seeded_shuffle(x) = Random.shuffle(Random.default_rng(), x)
    end
    return nothing
end

install_deterministic_erpgen_rng!()

function dataset_matched_config(cfg::ERPGenS.GenerationConfig)
    sim = ERPGenS.SimulationConfig(
        mu_dist = cfg.sim.mu_dist,
        sigma_dist = cfg.sim.sigma_dist,
        epoch_duration_dist = SS.fixed_dist(REAL_EPOCH_DURATION_S),
        sampling_rate_dist = SS.fixed_dist(REAL_SAMPLING_RATE),
        n_trials_dist = SS.fixed_dist(Float64(REAL_N_TRIALS)),
    )
    processing = ERPGenS.ProcessingConfig(
        dropout_trials_rate_dist = SS.fixed_dist(0.0),
        crop_start_dist = SS.fixed_dist(0.0),
        crop_end_dist = SS.fixed_dist(0.0),
        zscore_timepoints = true,
        resize_antialias = true,
        low_pass_factor = LOW_PASS_FACTOR,
        resize_method = cfg.processing.resize_method,
        target_height = TARGET_SIZE[1],
        target_width = TARGET_SIZE[2],
    )
    runtime = ERPGenS.RuntimeConfig(
        threaded = true,
        show_progress = false,
        blas_threads = 1,
        progress_every = 50,
    )
    return ERPGenS.GenerationConfig(
        sim = sim,
        components = cfg.components,
        patterns = cfg.patterns,
        noise = cfg.noise,
        processing = processing,
        runtime = runtime,
    )
end

base_cfg_from_helper = SS.make_base_config(target_size = TARGET_SIZE, apply_lowpass = true)
base_cfg = dataset_matched_config(base_cfg_from_helper)
param_specs = SS.parameter_specs(base_cfg)
param_symbols = SS.parameter_symbols(base_cfg)

@assert Int(round(mean(base_cfg.sim.n_trials_dist))) == REAL_N_TRIALS
@assert Int(round(mean(base_cfg.sim.epoch_duration_dist) * mean(base_cfg.sim.sampling_rate_dist))) + 1 == REAL_N_TIMEPOINTS
@assert length(param_specs) == 24
@assert length(param_symbols) == 48

function top_row_for_spec(top_df::DataFrame, spec)
    idx = findfirst((top_df.search_method .== spec.search_method) .& (Int.(top_df.candidate_index) .== spec.candidate_index))
    idx === nothing && error("Missing top_per_strategy row for $(spec.search_method) candidate $(spec.candidate_index).")
    return top_df[idx, :]
end

function dists_from_top_row(row)
    dists = Dict{Symbol, Distribution}()
    for spec in param_specs
        mu = Float64(row[spec.mean_symbol])
        sigma = max(Float64(row[spec.std_symbol]), 1e-6)
        dists[spec.key] = Normal(mu, sigma)
    end
    return dists
end

function config_from_top_row(row)
    dists = dists_from_top_row(row)
    return SS.parameterized_config(base_cfg;
        mu_dist = dists[:sim_mu],
        sigma_dist = dists[:sim_sigma],
        p100_width_dist = dists[:p100_width],
        p100_n170_gap_dist = dists[:p100_n170_gap],
        n170_p300_gap_dist = dists[:n170_p300_gap],
        n170_width_dist = dists[:n170_width],
        p300_width_dist = dists[:p300_width],
        p1_beta_dist = dists[:p1_beta],
        p3_beta_dist = dists[:p3_beta],
        n1_beta1_dist = dists[:n1_beta1],
        n1_beta2_dist = dists[:n1_beta2],
        n1_beta3_dist = dists[:n1_beta3],
        componentA_amp_dist = dists[:componentA_amp],
        componentB_amp_dist = dists[:componentB_amp],
        componentC_amp_dist = dists[:componentC_amp],
        tilted_bar_hanning_length_dist = dists[:tilted_bar_hanning_length],
        one_sided_fan_duration_divisor_dist = dists[:one_sided_fan_duration_divisor],
        one_sided_fan_log_mu_offset_dist = dists[:one_sided_fan_log_mu_offset],
        one_sided_fan_log_sigma_dist = dists[:one_sided_fan_log_sigma],
        one_sided_fan_support_max_dist = dists[:one_sided_fan_support_max],
        pink_noise_dist = dists[:noise_pink],
        white_noise_dist = dists[:noise_white],
        red_noise_dist = dists[:noise_red],
        exponential_noise_dist = dists[:noise_exponential],
    )
end

function params_namedtuple_from_row(row)
    pairs = Pair{Symbol, Float64}[]
    for sym in param_symbols
        push!(pairs, sym => Float64(row[sym]))
    end
    return (; pairs...)
end

isfile(TOP_PER_STRATEGY_CSV) || error("Missing top strategy CSV: $(TOP_PER_STRATEGY_CSV)")
top_per_strategy_df = CSV.read(TOP_PER_STRATEGY_CSV, DataFrame)

candidate_records = NamedTuple[]
for spec in SIM_MODEL_SPECS
    row = top_row_for_spec(top_per_strategy_df, spec)
    push!(candidate_records, merge(spec, (
        cfg = config_from_top_row(row),
        params = params_namedtuple_from_row(row),
        source_balanced_accuracy_mean = Float64(row.balanced_accuracy_mean),
        source_macro_f1_mean = Float64(row.macro_f1_mean),
    )))
end

candidate_overview_df = DataFrame([
    (
        model_kind = r.model_kind,
        search_method = r.search_method,
        candidate_index = r.candidate_index,
        source_balanced_accuracy_mean = r.source_balanced_accuracy_mean,
        source_macro_f1_mean = r.source_macro_f1_mean,
    )
    for r in candidate_records
])
show(candidate_overview_df, allrows = true, allcols = true)
println()

Reference dimensions before downscale: trials=2508, timepoints=513, sampling_rate=512.0, epoch_duration_s=1.0
3×5 DataFrame
 Row │ model_kind           search_method    candidate_index  source_balanced_accuracy_mean  source_macro_f1_mean 
     │ String               String           Int64            Float64                        Float64              
─────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ sim_broad_random     broad_random                  10                       0.926196              0.797011
   2 │ sim_latin_hypercube  latin_hypercube               21                       0.915666              0.78209
   3 │ sim_monte_carlo      monte_carlo                   38                       0.904393              0.85221


In [5]:
# ============================================================================
# Fixed-trial augmented simulated-data generation with true threading
# ============================================================================
# ERPGen.generate_erp_images is deliberately not used here because H1.1b now
# follows the Week-23 model-input contract: split each parent into fixed
# target-trial chunks, keep all sigmoid chunks, keep one no-class chunk, then
# materialize the four inverse-sort/polarity variants before z-score ->
# Gaussian smoothing -> resize. The outer pair loop is threaded with
# preallocated output arrays and per-iteration seeds.

const SIM_CHUNKS_PER_RAW_IMAGE = length(target_trial_mod_chunks(
    collect(1:REAL_N_TRIALS),
    TARGET_TRIALS;
    source_row_id = 1,
    sort_variable = "simulated",
))
const SIM_NO_CLASS_CHUNKS_PER_RAW_IMAGE = min(NO_CLASS_CHUNKS_PER_ORIGIN, SIM_CHUNKS_PER_RAW_IMAGE)
const SIM_IMAGES_PER_PAIR = (SIM_CHUNKS_PER_RAW_IMAGE + SIM_NO_CLASS_CHUNKS_PER_RAW_IMAGE) * AUGMENTATION_VARIANT_COUNT

function simulated_order(events::DataFrame, pattern::Symbol, rng::AbstractRNG)
    if pattern == :sigmoid
        return sortperm(collect(zip(events[!, ERPGenS.DELTA_LATENCY], events.latency)))
    elseif pattern == :no_class
        return randperm(rng, nrow(events))
    end
    error("Unsupported simulated pattern: $(pattern)")
end

function simulate_raw_pair_inputs(cfg::ERPGenS.GenerationConfig, seed::Integer)
    Random.seed!(seed)
    rng = Random.Xoshiro(seed)
    raw = ERPGenS.simulate_raw_erp(cfg, rng)
    dropped = ERPGenS.apply_trial_dropout(raw.data, raw.events, cfg.processing, rng)
    cropped = ERPGenS.apply_cropping(dropped.data, cfg.processing, rng, raw.params.sampling_rate)
    data_time_trials = Float32.(cropped.data)
    events_trials = dropped.events
    @assert size(data_time_trials) == (REAL_N_TIMEPOINTS, REAL_N_TRIALS) "Simulated dimensions do not match the fixation reference dimensions."
    return data_time_trials, events_trials
end

function augmented_images_for_pattern(data_time_trials::AbstractMatrix, events::DataFrame, pattern::Symbol, seed::Integer;
        source_row_id::Int)
    rng = Random.Xoshiro(seed)
    order = simulated_order(events, pattern, rng)
    chunks = target_trial_mod_chunks(
        order,
        TARGET_TRIALS;
        source_row_id = source_row_id,
        sort_variable = String(pattern),
    )
    chunk_idxs = pattern == :sigmoid ? collect(eachindex(chunks)) : no_class_chunk_indices(chunks, source_row_id)
    images = Matrix{Float32}[]
    variants = Symbol[]
    chunk_indices = Int[]
    chunk_roles = String[]
    mod_variants = String[]

    for chunk_idx in chunk_idxs
        chunk = chunks[chunk_idx]
        data_part = Float32.(data_time_trials[:, chunk.trial_indices])
        mod_variant = chunk.reused_fill_count == 0 ?
            @sprintf("modtarget_%04d_part%03d", TARGET_TRIALS, chunk.chunk_index) :
            @sprintf("modtarget_%04d_remainder%03d_fill%03d", TARGET_TRIALS, chunk.chunk_index, chunk.reused_fill_count)

        for augmentation in AUGMENTATION_VARIANTS
            push!(images, preprocess_ordered_chunk_variant(data_part, augmentation))
            push!(variants, augmentation.name)
            push!(chunk_indices, Int(chunk.chunk_index))
            push!(chunk_roles, String(chunk.chunk_role))
            push!(mod_variants, mod_variant)
        end
    end
    return images, variants, chunk_indices, chunk_roles, mod_variants
end

function simulate_augmented_pair_images(cfg::ERPGenS.GenerationConfig, seed::Integer; pair_index::Int = 1)
    data_time_trials, events = simulate_raw_pair_inputs(cfg, seed)
    imgs = Matrix{Float32}[]
    labels = Int[]
    variants = Symbol[]
    chunk_indices = Int[]
    chunk_roles = String[]
    mod_variants = String[]
    patterns = String[]

    for spec in ((pattern = :sigmoid, label = 1, seed_offset = 101, source_row_id = 2 * pair_index - 1),
            (pattern = :no_class, label = 0, seed_offset = 202, source_row_id = 2 * pair_index))
        pattern_imgs, pattern_variants, pattern_chunk_indices, pattern_chunk_roles, pattern_mod_variants = augmented_images_for_pattern(
            data_time_trials,
            events,
            spec.pattern,
            seed + spec.seed_offset;
            source_row_id = spec.source_row_id,
        )
        append!(imgs, pattern_imgs)
        append!(labels, fill(spec.label, length(pattern_imgs)))
        append!(variants, pattern_variants)
        append!(chunk_indices, pattern_chunk_indices)
        append!(chunk_roles, pattern_chunk_roles)
        append!(mod_variants, pattern_mod_variants)
        append!(patterns, fill(String(spec.pattern), length(pattern_imgs)))
    end

    @assert length(imgs) == SIM_IMAGES_PER_PAIR "Unexpected simulated images per pair: $(length(imgs)) != $(SIM_IMAGES_PER_PAIR)."
    return imgs, labels, variants, chunk_indices, chunk_roles, mod_variants, patterns
end

function generate_augmented_simulated_dataset(cfg::ERPGenS.GenerationConfig; n_pairs::Int, seed::Integer, threaded::Bool = true, shuffle_dataset::Bool = true)
    total = n_pairs * SIM_IMAGES_PER_PAIR
    imgs = Vector{Matrix{Float32}}(undef, total)
    labels = Vector{Int}(undef, total)
    pair_indices = Vector{Int}(undef, total)
    variant_names = Vector{String}(undef, total)
    chunk_indices = Vector{Int}(undef, total)
    chunk_roles = Vector{String}(undef, total)
    mod_variants = Vector{String}(undef, total)
    patterns = Vector{String}(undef, total)

    function fill_pair!(i::Int)
        pair_seed = derived_seed(:sim_pair, seed, i)
        pair_imgs, pair_labels, pair_variants, pair_chunk_indices, pair_chunk_roles, pair_mod_variants, pair_patterns = simulate_augmented_pair_images(cfg, pair_seed; pair_index = i)
        base = (i - 1) * SIM_IMAGES_PER_PAIR
        for j in 1:SIM_IMAGES_PER_PAIR
            k = base + j
            imgs[k] = pair_imgs[j]
            labels[k] = pair_labels[j]
            pair_indices[k] = i
            variant_names[k] = String(pair_variants[j])
            chunk_indices[k] = pair_chunk_indices[j]
            chunk_roles[k] = pair_chunk_roles[j]
            mod_variants[k] = pair_mod_variants[j]
            patterns[k] = pair_patterns[j]
        end
        return nothing
    end

    if threaded
        Threads.@threads :static for i in 1:n_pairs
            fill_pair!(i)
        end
    else
        for i in 1:n_pairs
            fill_pair!(i)
        end
    end

    order = collect(1:total)
    if shuffle_dataset
        rng = Random.Xoshiro(derived_seed(:shuffle_sim_dataset, seed))
        order = randperm(rng, total)
    end

    out = DataFrame(
        sample_id = collect(1:total),
        pair_index = pair_indices[order],
        pattern = patterns[order],
        mod_variant = mod_variants[order],
        chunk_index = chunk_indices[order],
        chunk_role = chunk_roles[order],
        variant = variant_names[order],
        binary_label = labels[order],
    )
    out.processed_img = imgs[order]
    return out
end

function threading_mini_test(cfg::ERPGenS.GenerationConfig; n_pairs::Int = MINI_TEST_PAIRS)
    Threads.nthreads() >= MIN_GENERATION_THREADS || error(
        "Threaded generation requires at least $(MIN_GENERATION_THREADS) Julia threads for this run. " *
        "Restart with JULIA_NUM_THREADS=$(MIN_GENERATION_THREADS) or higher. Current Threads.nthreads()=$(Threads.nthreads())."
    )
    println("Threading mini-test with $(n_pairs) simulated pairs and Threads.nthreads()=$(Threads.nthreads()).")
    serial_time_s = @elapsed begin
        _ = generate_augmented_simulated_dataset(cfg; n_pairs = n_pairs, seed = derived_seed(:mini_serial), threaded = false, shuffle_dataset = false)
    end
    cleanup_device!()
    threaded_time_s = @elapsed begin
        _ = generate_augmented_simulated_dataset(cfg; n_pairs = n_pairs, seed = derived_seed(:mini_threaded), threaded = true, shuffle_dataset = false)
    end
    cleanup_device!()
    speedup = serial_time_s / max(threaded_time_s, eps(Float64))
    println("Generation mini-test | serial=", round(serial_time_s, digits = 2), "s | threaded=",
        round(threaded_time_s, digits = 2), "s | speedup=", round(speedup, digits = 2),
        "x | thread_count=", Threads.nthreads())
    return (
        n_pairs = n_pairs,
        serial_time_s = Float64(serial_time_s),
        threaded_time_s = Float64(threaded_time_s),
        speedup = Float64(speedup),
        threads = Threads.nthreads(),
    )
end

threading_mini_test (generic function with 1 method)

In [6]:
# ============================================================================
# Real-fixations baseline training materialization: 200-trial chunks + variants
# ============================================================================

# Shared fixed-trial chunking and inverse-sort/polarity preprocessing helpers
# are defined above so real validation, simulated train/holdout, and this
# real-fixations baseline all materialize model inputs the same way.

function select_real_baseline_training_labels(labels::DataFrame)
    keep = [
        String(row.erp_class) == NO_CLASS ||
        (String(row.erp_class) == SIGMOID_CLASS && String(row.sort_variable) == REAL_BASELINE_SIGMOID_SORT_VARIABLE)
        for row in eachrow(labels)
    ]
    selected = copy(labels[keep, :])
    selected.binary_label = Int.(selected.erp_class .== SIGMOID_CLASS)
    sort!(selected, [:erp_class, :sort_variable, :channel_name])
    selected.source_row_id = collect(1:nrow(selected))
    count(==(1), selected.binary_label) > 0 || error("No real baseline sigmoid labels after filtering.")
    count(==(0), selected.binary_label) > 0 || error("No real baseline no_class labels after filtering.")
    return selected
end

function materialize_real_fixations_training_set(; target_trials::Int = TARGET_TRIALS)
    events, labels, _ = load_dataset_tables(REAL_BASELINE_SOURCE)
    selected = select_real_baseline_training_labels(labels)
    rows = NamedTuple[]
    images = Matrix{Float32}[]
    signal_cache = Dict{Tuple{String, String}, Matrix{Float32}}()
    base_sample_id = 0

    for row in eachrow(selected)
        data_time_trials = load_signal_for_label(REAL_BASELINE_SOURCE, row.channel_name, signal_cache)
        sort_col = Symbol(String(row.sort_variable))
        filtered = filtered_events_and_data_for_sort(data_time_trials, events, sort_col)
        nrow(filtered.events) >= target_trials || error("$(REAL_BASELINE_SOURCE)/$(row.channel_name)/$(row.sort_variable) has only $(nrow(filtered.events)) valid trials; target_trials=$(target_trials).")
        order = sorted_order_for_variant(filtered.events, sort_col)
        chunks = target_trial_mod_chunks(
            order,
            target_trials;
            source_row_id = Int(row.source_row_id),
            sort_variable = String(row.sort_variable),
        )
        chunk_idxs = chunk_indices_for_label(Int(row.binary_label), chunks, Int(row.source_row_id))
        for chunk_idx in chunk_idxs
            chunk = chunks[chunk_idx]
            idxs = chunk.trial_indices
            events_part = filtered.events[idxs, :]
            data_part = filtered.data_time_trials[:, idxs]
            base_sample_id += 1
            mod_variant = chunk.reused_fill_count == 0 ?
                @sprintf("modtarget_%04d_part%03d", target_trials, chunk.chunk_index) :
                @sprintf("modtarget_%04d_remainder%03d_fill%03d", target_trials, chunk.chunk_index, chunk.reused_fill_count)
            for (augmentation_variant_index, augmentation) in enumerate(AUGMENTATION_VARIANTS)
                img = preprocess_fixed_trial_image(data_part, events_part, sort_col, augmentation)
                push!(rows, (
                    sample_id = length(rows) + 1,
                    base_sample_id = base_sample_id,
                    dataset_key = REAL_BASELINE_SOURCE,
                    channel_name = String(row.channel_name),
                    sort_variable = String(row.sort_variable),
                    erp_class = String(row.erp_class),
                    binary_label = Int(row.binary_label),
                    target_trials = Int(target_trials),
                    chunk_index = Int(chunk.chunk_index),
                    chunk_count = Int(chunk.chunk_count),
                    chunk_role = String(chunk.chunk_role),
                    reused_fill_count = Int(chunk.reused_fill_count),
                    n_filtered_out = Int(filtered.n_filtered_out),
                    mod_variant = mod_variant,
                    augmentation_variant_index = Int(augmentation_variant_index),
                    augmentation_name = String(augmentation.name),
                    augmentation_label = String(augmentation.label),
                    inverse_sort = Bool(augmentation.inverse_sort),
                    inverse_polarity = Bool(augmentation.inverse_polarity),
                    variant = String(augmentation.name),
                ))
                push!(images, img)
            end
        end
    end

    out = DataFrame(rows)
    out.processed_img = images
    @assert all(size.(out.processed_img) .== Ref(TARGET_SIZE))
    return out
end

materialize_real_fixations_training_set (generic function with 1 method)

In [7]:
# ============================================================================
# Model training and evaluation
# ============================================================================

function build_resnet18_pretrained_for_run()
    model, matched = CNNUtils.build_resnet18_single_channel_pretrained(n_classes = 2, in_channels = 1)
    return model, Int(matched)
end

function train_resnet18!(model, X_train::Array{Float32, 4}, y_train::Vector{Int}, profile; run_tag::AbstractString, seed::Integer)
    set_all_seeds!(seed)
    y_train_oh = onehotbatch(y_train, 0:1) |> Array{Float32}
    y_train_oh = smooth_onehot(y_train_oh, profile.label_smoothing)
    train_loader = DataLoader((X_train, y_train_oh); batchsize = profile.batchsize, shuffle = true)
    model = device(model)
    opt_state = Flux.setup(Flux.Adam(profile.lr), model)
    class_weights = device(reshape(Float32.(profile.class_weights), :, 1))
    Flux.trainmode!(model)

    train_time_s = @elapsed begin
        for epoch in 1:profile.nepochs
            running_loss = 0f0
            n_batches = 0
            epoch_time_s = @elapsed begin
                for (xb_cpu, yb_cpu) in train_loader
                    xb = device(xb_cpu)
                    yb = device(yb_cpu)
                    loss_val, grads = Flux.withgradient(model) do m
                        weighted_logitcrossentropy(m(xb), yb, class_weights)
                    end
                    opt_state, model = Flux.update!(opt_state, model, grads[1])
                    running_loss += Float32(loss_val)
                    n_batches += 1
                end
            end
            @info "$(run_tag) | epoch $(epoch)/$(profile.nepochs) | loss=$(@sprintf("%.5f", Float64(running_loss / max(1, n_batches)))) | epoch_time_s=$(@sprintf("%.2f", epoch_time_s))"
        end
    end
    return model, Float64(train_time_s)
end

function evaluate_resnet18(model, X_val::Array{Float32, 4}, y_val::Vector{Int}; batchsize::Int = PREDICT_BATCHSIZE)
    Flux.testmode!(model, true)
    y_pred = Int[]
    classification_time_s = @elapsed begin
        for start_idx in 1:batchsize:size(X_val, 4)
            idx = start_idx:min(start_idx + batchsize - 1, size(X_val, 4))
            logits = Array(cpu(model(device(X_val[:, :, :, idx]))))
            append!(y_pred, Int.(onecold(logits, 0:1)))
        end
    end
    metrics = binary_metrics(y_pred, y_val)
    return metrics, y_pred, Float64(classification_time_s)
end

function metric_row(model_kind::AbstractString, candidate_index_or_real, repeat_index::Int,
        validation_source::AbstractString, metrics, y_true::Vector{Int}, y_pred::Vector{Int};
        train_time_s::Real,
        generation_time_s::Real,
        classification_time_s::Real)
    n_validation = length(y_true)
    collapsed = prediction_collapsed(y_pred)
    return (
        model_kind = String(model_kind),
        candidate_index_or_real = string(candidate_index_or_real),
        repeat_index = Int(repeat_index),
        validation_source = String(validation_source),
        n_validation = Int(n_validation),
        n_sigmoid = count(==(1), y_true),
        n_no_class = count(==(0), y_true),
        balanced_accuracy = Float64(metrics.balanced_accuracy),
        macro_f1 = Float64(metrics.macro_f1),
        precision = Float64(metrics.precision),
        recall = Float64(metrics.recall),
        pred_count_0 = count(==(0), y_pred),
        pred_count_1 = count(==(1), y_pred),
        collapsed = Bool(collapsed),
        train_time_s = Float64(train_time_s),
        generation_time_s = Float64(generation_time_s),
        classification_time_s = Float64(classification_time_s),
    )
end

function evaluate_on_named_set!(rows::Vector{NamedTuple}, model, model_kind::AbstractString, candidate_index_or_real,
        repeat_index::Int, validation_source::AbstractString, X, y;
        train_time_s::Real,
        generation_time_s::Real)
    metrics, y_pred, classification_time_s = evaluate_resnet18(model, X, y; batchsize = PREDICT_BATCHSIZE)
    push!(rows, metric_row(model_kind, candidate_index_or_real, repeat_index, validation_source, metrics, y, y_pred;
        train_time_s = train_time_s,
        generation_time_s = generation_time_s,
        classification_time_s = classification_time_s,
    ))
    return metrics, y_pred
end

evaluate_on_named_set! (generic function with 1 method)

In [8]:
# ============================================================================
# Visual preview: one simulated sigmoid per strategy and one real sigmoid per source
# ============================================================================

function image_colorrange(img::AbstractMatrix)
    vmax = quantile(abs.(vec(Float64.(img))), 0.98)
    vmax = max(vmax, eps(Float64))
    return (-vmax, vmax)
end

function add_image_axis!(fig, row::Int, col::Int, img::AbstractMatrix, title::AbstractString)
    ax = Axis(fig[row, col]; title = title, xticks = [], yticks = [])
    heatmap!(ax, Float32.(img); colormap = :balance, colorrange = image_colorrange(img))
    hidespines!(ax)
    return ax
end

function first_real_sigmoid_image(validation_set)
    idx = findfirst(==(1), Int.(validation_set.df.binary_label))
    idx === nothing && error("Validation set has no sigmoid image.")
    return validation_set.df.processed_img[idx]
end

function save_preview_figure(candidate_records, real_validation_sets; output_path::AbstractString = PREVIEW_PATH)
    rows = length(REQUESTED_REAL_SOURCE_KEYS)
    cols = length(candidate_records) + 1
    fig = Figure(size = (330 * cols, 230 * rows), figure_padding = (12, 16, 12, 12))

    sim_preview = Dict{String, Matrix{Float32}}()
    for record in candidate_records
        imgs, labels, _ = simulate_augmented_pair_images(record.cfg, derived_seed(:preview, record.model_kind))
        sigmoid_idx = findfirst(==(1), labels)
        sim_preview[record.model_kind] = imgs[sigmoid_idx]
    end

    for (r, source_key) in enumerate(REQUESTED_REAL_SOURCE_KEYS)
        for (c, record) in enumerate(candidate_records)
            add_image_axis!(fig, r, c, sim_preview[record.model_kind], "$(record.model_kind)")
        end
        add_image_axis!(fig, r, cols, first_real_sigmoid_image(real_validation_sets[source_key]), source_key)
        Label(fig[r, 0], source_key; rotation = pi / 2, tellheight = false, fontsize = 12)
    end

    colgap!(fig.layout, 8)
    rowgap!(fig.layout, 8)
    resize_to_layout!(fig)
    mkpath(dirname(output_path))
    save(output_path, fig)
    println("Saved preview figure: ", output_path)
    return fig
end

preview_fig = save_preview_figure(candidate_records, real_validation_sets; output_path = PREVIEW_PATH)
preview_fig

ArgumentError: ArgumentError: argument must not be empty

In [9]:
# ============================================================================
# Sanity gate, time estimate, experiment runners, and CSV exports
# ============================================================================

function print_time_estimate(probe, real_validation_sets)
    sim_models = length(candidate_records)
    train_generation_runs = sim_models * N_REPEATS
    train_generation_pairs = train_generation_runs * N_PER_PATTERN
    holdout_generation_pairs = train_generation_runs * SIM_HOLDOUT_PAIRS
    total_generation_pairs = train_generation_pairs + holdout_generation_pairs

    known_serial_train_s = sim_models * N_REPEATS * 311.7
    observed_per_1000_pairs_s = probe.threaded_time_s * (1000 / probe.n_pairs)
    observed_generation_estimate_s = observed_per_1000_pairs_s * (total_generation_pairs / 1000)

    n_real_validation_images = sum(length(v.y) for v in values(real_validation_sets))
    n_sim_holdout_images_per_repeat = SIM_HOLDOUT_PAIRS * SIM_IMAGES_PER_PAIR
    n_inference_images = train_generation_runs * (n_real_validation_images + n_sim_holdout_images_per_repeat) +
                         N_REPEATS * n_real_validation_images

    println("Time estimate before productive run:")
    println("  Known serial sim-training generation: 3 strategies x 3 repeats x 311.7s = ", round(known_serial_train_s / 60, digits = 1), " min")
    println("  Observed threaded generation estimate including holdouts: ", round(observed_generation_estimate_s / 60, digits = 1), " min for ", total_generation_pairs, " pairs")
    println("  Real-source inference load: ", n_inference_images, " images across sim and real-baseline repeats; training time is expected to be smaller than generation.")
    return (
        known_serial_train_s = known_serial_train_s,
        observed_generation_estimate_s = observed_generation_estimate_s,
        n_inference_images = n_inference_images,
    )
end

function sanity_gate!(record)
    println("\n=== Sanity gate: $(record.model_kind) candidate $(record.candidate_index) ===")
    sanity_seed = derived_seed(:sanity, record.model_kind, record.candidate_index)
    train_generation_time_s = @elapsed train_df = generate_augmented_simulated_dataset(
        record.cfg;
        n_pairs = N_PER_PATTERN,
        seed = sanity_seed,
        threaded = true,
        shuffle_dataset = true,
    )
    holdout_generation_time_s = @elapsed holdout_df = generate_augmented_simulated_dataset(
        record.cfg;
        n_pairs = SIM_HOLDOUT_PAIRS,
        seed = derived_seed(:sanity_holdout, record.model_kind, record.candidate_index),
        threaded = true,
        shuffle_dataset = false,
    )
    X_train = CNNUtils.images_to_tensor(train_df.processed_img)
    y_train = Int.(train_df.binary_label)
    X_holdout = CNNUtils.images_to_tensor(holdout_df.processed_img)
    y_holdout = Int.(holdout_df.binary_label)

    set_all_seeds!(derived_seed(:sanity_model_init, record.model_kind))
    model, _ = build_resnet18_pretrained_for_run()
    model, train_time_s = train_resnet18!(model, X_train, y_train, TRAINING_PROFILE;
        run_tag = "sanity/$(record.model_kind)",
        seed = derived_seed(:sanity_train, record.model_kind),
    )
    metrics, y_pred, classification_time_s = evaluate_resnet18(model, X_holdout, y_holdout; batchsize = PREDICT_BATCHSIZE)
    min_pred_frac = min(count(==(0), y_pred), count(==(1), y_pred)) / length(y_pred)
    println("Sanity holdout | BAcc=", round(metrics.balanced_accuracy, digits = 4),
        " | macro_F1=", round(metrics.macro_f1, digits = 4),
        " | pred0=", count(==(0), y_pred),
        " | pred1=", count(==(1), y_pred),
        " | min_pred_frac=", round(min_pred_frac, digits = 4),
        " | generation_time_s=", round(train_generation_time_s + holdout_generation_time_s, digits = 2),
        " | train_time_s=", round(train_time_s, digits = 2),
        " | classification_time_s=", round(classification_time_s, digits = 2))

    cleanup_device!()
    if metrics.balanced_accuracy < SANITY_BACC_MIN || min_pred_frac < SANITY_CLASS_BALANCE_FRAC
        error("Sanity gate failed: sim_holdout BAcc=$(metrics.balanced_accuracy), min predicted class fraction=$(min_pred_frac).")
    end
    return true
end

function metadata_row_base(model_kind::AbstractString, candidate_index_or_real, repeat_index::Int; extra_pairs = Pair{Symbol, Any}[])
    pairs = Pair{Symbol, Any}[
        :model_kind => String(model_kind),
        :candidate_index_or_real => string(candidate_index_or_real),
        :repeat_index => Int(repeat_index),
        :threads => Threads.nthreads(),
        :use_gpu => Bool(USE_GPU),
        :gpu_name => String(GPU_NAME),
        :training_profile => String(TRAINING_PROFILE.name),
        :train_epochs => TRAINING_PROFILE.nepochs,
        :train_batchsize => TRAINING_PROFILE.batchsize,
        :train_lr => Float64(TRAINING_PROFILE.lr),
        :label_smoothing => Float64(TRAINING_PROFILE.label_smoothing),
        :target_size => "$(TARGET_SIZE[1])x$(TARGET_SIZE[2])",
        :target_trials_real_baseline => TARGET_TRIALS,
        :timestamp => string(now()),
    ]
    append!(pairs, extra_pairs)
    return (; pairs...)
end

function run_sim_repeats(record, real_validation_sets)
    metric_rows = NamedTuple[]
    metadata_rows = NamedTuple[]
    for repeat_index in 1:N_REPEATS
        repeat_seed = derived_seed(:sim_repeat, record.model_kind, repeat_index)
        train_seed = derived_seed(:sim_train_generation, record.model_kind, repeat_index)
        holdout_seed = derived_seed(:sim_holdout_generation, record.model_kind, repeat_index)
        model_seed = derived_seed(:model_init, record.model_kind, repeat_index)
        run_tag = "$(record.model_kind)/candidate$(record.candidate_index)/repeat$(repeat_index)"
        println("\n=== $(run_tag) ===")

        train_generation_time_s = @elapsed train_df = generate_augmented_simulated_dataset(
            record.cfg;
            n_pairs = N_PER_PATTERN,
            seed = train_seed,
            threaded = true,
            shuffle_dataset = true,
        )
        holdout_generation_time_s = @elapsed holdout_df = generate_augmented_simulated_dataset(
            record.cfg;
            n_pairs = SIM_HOLDOUT_PAIRS,
            seed = holdout_seed,
            threaded = true,
            shuffle_dataset = false,
        )
        generation_time_s = train_generation_time_s + holdout_generation_time_s
        X_train = CNNUtils.images_to_tensor(train_df.processed_img)
        y_train = Int.(train_df.binary_label)
        X_holdout = CNNUtils.images_to_tensor(holdout_df.processed_img)
        y_holdout = Int.(holdout_df.binary_label)

        set_all_seeds!(model_seed)
        model, pretrained_loaded = build_resnet18_pretrained_for_run()
        model, train_time_s = train_resnet18!(model, X_train, y_train, TRAINING_PROFILE;
            run_tag = run_tag,
            seed = derived_seed(:train_loop, record.model_kind, repeat_index),
        )

        evaluate_on_named_set!(metric_rows, model, record.model_kind, record.candidate_index,
            repeat_index, "sim_holdout", X_holdout, y_holdout;
            train_time_s = train_time_s,
            generation_time_s = generation_time_s)

        for source_key in REQUESTED_REAL_SOURCE_KEYS
            val = real_validation_sets[source_key]
            evaluate_on_named_set!(metric_rows, model, record.model_kind, record.candidate_index,
                repeat_index, source_key, val.X, val.y;
                train_time_s = train_time_s,
                generation_time_s = generation_time_s)
        end

        param_pairs = [sym => getfield(record.params, sym) for sym in propertynames(record.params)]
        push!(metadata_rows, metadata_row_base(record.model_kind, record.candidate_index, repeat_index;
            extra_pairs = vcat(Pair{Symbol, Any}[
                :search_method => String(record.search_method),
                :train_seed => train_seed,
                :holdout_seed => holdout_seed,
                :model_seed => model_seed,
                :repeat_seed => repeat_seed,
                :n_train_images => length(y_train),
                :n_train_sigmoid => count(==(1), y_train),
                :n_train_no_class => count(==(0), y_train),
                :n_holdout_images => length(y_holdout),
                :pretrained_params_loaded => pretrained_loaded,
                :train_generation_time_s => Float64(train_generation_time_s),
                :holdout_generation_time_s => Float64(holdout_generation_time_s),
                :train_time_s => Float64(train_time_s),
            ], param_pairs),
        ))
        cleanup_device!()
    end
    return DataFrame(metric_rows), DataFrame(metadata_rows)
end

function run_real_baseline_repeats(real_train_df::DataFrame, real_train_generation_time_s::Real, real_validation_sets)
    metric_rows = NamedTuple[]
    metadata_rows = NamedTuple[]
    X_train = CNNUtils.images_to_tensor(real_train_df.processed_img)
    y_train = Int.(real_train_df.binary_label)
    for repeat_index in 1:N_REPEATS
        model_seed = derived_seed(:real_baseline_model_init, repeat_index)
        run_tag = "real_fixations_baseline/repeat$(repeat_index)"
        println("\n=== $(run_tag) ===")
        set_all_seeds!(model_seed)
        model, pretrained_loaded = build_resnet18_pretrained_for_run()
        model, train_time_s = train_resnet18!(model, X_train, y_train, TRAINING_PROFILE;
            run_tag = run_tag,
            seed = derived_seed(:real_baseline_train_loop, repeat_index),
        )
        for source_key in REQUESTED_REAL_SOURCE_KEYS
            val = real_validation_sets[source_key]
            evaluate_on_named_set!(metric_rows, model, "real_fixations_baseline", "real",
                repeat_index, source_key, val.X, val.y;
                train_time_s = train_time_s,
                generation_time_s = real_train_generation_time_s)
        end
        push!(metadata_rows, metadata_row_base("real_fixations_baseline", "real", repeat_index;
            extra_pairs = Pair{Symbol, Any}[
                :search_method => "real_fixations_baseline",
                :train_seed => derived_seed(:real_baseline_train_loop, repeat_index),
                :holdout_seed => missing,
                :model_seed => model_seed,
                :repeat_seed => derived_seed(:real_baseline_repeat, repeat_index),
                :n_train_images => length(y_train),
                :n_train_sigmoid => count(==(1), y_train),
                :n_train_no_class => count(==(0), y_train),
                :n_holdout_images => missing,
                :pretrained_params_loaded => pretrained_loaded,
                :train_generation_time_s => Float64(real_train_generation_time_s),
                :holdout_generation_time_s => 0.0,
                :train_time_s => Float64(train_time_s),
            ],
        ))
        cleanup_device!()
    end
    return DataFrame(metric_rows), DataFrame(metadata_rows)
end

function summarize_metrics(metrics_df::DataFrame)
    rows = NamedTuple[]
    for sdf in groupby(metrics_df, [:model_kind, :validation_source])
        valid = sdf[.!Bool.(coalesce.(sdf.collapsed, true)), :]
        push!(rows, (
            model_kind = String(sdf.model_kind[1]),
            validation_source = String(sdf.validation_source[1]),
            balanced_accuracy_mean = mean_or_missing(valid.balanced_accuracy),
            balanced_accuracy_std = std_or_zero(valid.balanced_accuracy),
            macro_f1_mean = mean_or_missing(valid.macro_f1),
            macro_f1_std = std_or_zero(valid.macro_f1),
            precision_mean = mean_or_missing(valid.precision),
            recall_mean = mean_or_missing(valid.recall),
            n_valid_repeats = nrow(valid),
            n_collapsed_repeats = count(Bool.(coalesce.(sdf.collapsed, true))),
            train_time_mean_s = mean_or_missing(sdf.train_time_s),
            generation_time_mean_s = mean_or_missing(sdf.generation_time_s),
        ))
    end
    out = DataFrame(rows)
    isempty(out) || sort!(out, [:model_kind, :validation_source])
    return out
end

function h1_1b_gap_summary(summary_df::DataFrame)
    rows = NamedTuple[]
    sim_model_kinds = [r.model_kind for r in candidate_records]
    for model_kind in sim_model_kinds
        sim_idx = findfirst((summary_df.model_kind .== model_kind) .& (summary_df.validation_source .== "sim_holdout"))
        sim_idx === nothing && continue
        sim_row = summary_df[sim_idx, :]
        for real_source in REQUESTED_REAL_SOURCE_KEYS
            real_idx = findfirst((summary_df.model_kind .== model_kind) .& (summary_df.validation_source .== real_source))
            real_idx === nothing && continue
            real_row = summary_df[real_idx, :]
            push!(rows, (
                model_kind = String(model_kind),
                real_source = String(real_source),
                sim_holdout_bacc_mean = sim_row.balanced_accuracy_mean,
                real_bacc_mean = real_row.balanced_accuracy_mean,
                gap_bacc_mean = sim_row.balanced_accuracy_mean - real_row.balanced_accuracy_mean,
                sim_holdout_f1_mean = sim_row.macro_f1_mean,
                real_f1_mean = real_row.macro_f1_mean,
                gap_f1_mean = sim_row.macro_f1_mean - real_row.macro_f1_mean,
            ))
        end
    end
    out = DataFrame(rows)
    isempty(out) || sort!(out, [:model_kind, :real_source])
    return out
end

function write_experiment_outputs(metrics_df::DataFrame, metadata_df::DataFrame)
    ensure_output_dir!()
    summary_df = summarize_metrics(metrics_df)
    gap_df = h1_1b_gap_summary(summary_df)
    CSV.write(joinpath(OUTPUT_DIR, "metrics_per_run.csv"), metrics_df)
    CSV.write(joinpath(OUTPUT_DIR, "metrics_summary.csv"), summary_df)
    CSV.write(joinpath(OUTPUT_DIR, "h1_1b_gap_summary.csv"), gap_df)
    CSV.write(joinpath(OUTPUT_DIR, "run_metadata.csv"), metadata_df)
    println("Wrote outputs to ", OUTPUT_DIR)
    return summary_df, gap_df
end

function run_experiment()
    ensure_output_dir!()
    probe = threading_mini_test(candidate_records[1].cfg; n_pairs = MINI_TEST_PAIRS)
    time_estimate = print_time_estimate(probe, real_validation_sets)
    sanity_gate!(candidate_records[1])

    println("\nMaterializing real-fixations baseline training set.")
    real_train_generation_time_s = @elapsed real_train_df = materialize_real_fixations_training_set(target_trials = TARGET_TRIALS)
    println("Real baseline training images: ", nrow(real_train_df),
        " | sigmoid=", count(==(1), Int.(real_train_df.binary_label)),
        " | no_class=", count(==(0), Int.(real_train_df.binary_label)),
        " | generation_time_s=", round(real_train_generation_time_s, digits = 2))

    metric_parts = DataFrame[]
    metadata_parts = DataFrame[]
    for record in candidate_records
        metrics_df, metadata_df = run_sim_repeats(record, real_validation_sets)
        push!(metric_parts, metrics_df)
        push!(metadata_parts, metadata_df)
    end
    real_metrics_df, real_metadata_df = run_real_baseline_repeats(real_train_df, real_train_generation_time_s, real_validation_sets)
    push!(metric_parts, real_metrics_df)
    push!(metadata_parts, real_metadata_df)

    metrics_df = vcat(metric_parts...; cols = :union)
    metadata_df = vcat(metadata_parts...; cols = :union)
    summary_df, gap_df = write_experiment_outputs(metrics_df, metadata_df)
    return (
        metrics_df = metrics_df,
        summary_df = summary_df,
        gap_df = gap_df,
        metadata_df = metadata_df,
        threading_probe = probe,
        time_estimate = time_estimate,
    )
end

results = run_experiment()
show(results.summary_df, allrows = true, allcols = true)
println("\n\nH1.1b gap summary:")
show(results.gap_df, allrows = true, allcols = true)
println()

Threading mini-test with 50 simulated pairs and Threads.nthreads()=16.
Generation mini-test | serial=15.98s | threaded=5.01s | speedup=3.19x | thread_count=16
Time estimate before productive run:
  Known serial sim-training generation: 3 strategies x 3 repeats x 311.7s = 46.8 min
  Observed threaded generation estimate including holdouts: 18.0 min for 10800 pairs
  Real-source inference load: 161952 images across sim and real-baseline repeats; training time is expected to be smaller than generation.

=== Sanity gate: sim_broad_random candidate 10 ===


┌ Info: sanity/sim_broad_random | epoch 1/8 | loss=0.20008 | epoch_time_s=49.65
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: sanity/sim_broad_random | epoch 2/8 | loss=0.14629 | epoch_time_s=8.16
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: sanity/sim_broad_random | epoch 3/8 | loss=0.12803 | epoch_time_s=8.13
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: sanity/sim_broad_random | epoch 4/8 | loss=0.11143 | epoch_time_s=8.15
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: sanity/sim_broad_random | epoch 5/8 | loss=0.09553 | epoch_time_s=8.15
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_noteb

Sanity holdout | BAcc=0.8211 | macro_F1=0.7952 | pred0=951 | pred1=10249 | min_pred_frac=0.0849 | generation_time_s=103.04 | train_time_s=106.78 | classification_time_s=0.65

Materializing real-fixations baseline training set.
Real baseline training images: 5260 | sigmoid=3224 | no_class=2036 | generation_time_s=5.9

=== sim_broad_random/candidate10/repeat1 ===


┌ Info: sim_broad_random/candidate10/repeat1 | epoch 1/8 | loss=0.19508 | epoch_time_s=8.40
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: sim_broad_random/candidate10/repeat1 | epoch 2/8 | loss=0.14598 | epoch_time_s=8.18
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: sim_broad_random/candidate10/repeat1 | epoch 3/8 | loss=0.12905 | epoch_time_s=8.19
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: sim_broad_random/candidate10/repeat1 | epoch 4/8 | loss=0.11351 | epoch_time_s=8.18
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: sim_broad_random/candidate10/repeat1 | epoch 5/8 | loss=0.09738 | epoch_time_s=8.17
└ @ Main


=== sim_broad_random/candidate10/repeat2 ===


...


=== real_fixations_baseline/repeat2 ===


┌ Info: real_fixations_baseline/repeat2 | epoch 1/8 | loss=0.57250 | epoch_time_s=0.79
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: real_fixations_baseline/repeat2 | epoch 2/8 | loss=0.19861 | epoch_time_s=0.79
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: real_fixations_baseline/repeat2 | epoch 3/8 | loss=0.10159 | epoch_time_s=0.78
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: real_fixations_baseline/repeat2 | epoch 4/8 | loss=0.07982 | epoch_time_s=0.79
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: real_fixations_baseline/repeat2 | epoch 5/8 | loss=0.08278 | epoch_time_s=0.78
└ @ Main /home/benjamin/Dokumente


=== real_fixations_baseline/repeat3 ===


┌ Info: real_fixations_baseline/repeat3 | epoch 1/8 | loss=0.57934 | epoch_time_s=0.78
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: real_fixations_baseline/repeat3 | epoch 2/8 | loss=0.22692 | epoch_time_s=0.80
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: real_fixations_baseline/repeat3 | epoch 3/8 | loss=0.12836 | epoch_time_s=0.78
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: real_fixations_baseline/repeat3 | epoch 4/8 | loss=0.09533 | epoch_time_s=0.79
└ @ Main /home/benjamin/Dokumente/BA2/notebooks/data_generation/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:36
┌ Info: real_fixations_baseline/repeat3 | epoch 5/8 | loss=0.08038 | epoch_time_s=0.78
└ @ Main /home/benjamin/Dokumente

Wrote outputs to /home/benjamin/Dokumente/BA2/notebooks/data_generation/outputs/cross_source_sim_to_real_sigmoid
23×12 DataFrame
 Row │ model_kind               validation_source              balanced_accuracy_mean  balanced_accuracy_std  macro_f1_mean   macro_f1_std      precision_mean  recall_mean     n_valid_repeats  n_collapsed_repeats  train_time_mean_s  generation_time_mean_s 
     │ String                   String                         Float64?                Float64?               Float64?        Float64?          Float64?        Float64?        Int64            Int64                Float64            Float64                
─────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ real_fixations_baseline  02_new_roamm_reading                         0.714799            0.0